# Experiment 12 — local sensitivity, constraint-aware allocation and engine replay

Based on [NB11](11-research-weighting-sizing.ipynb) and its [registered follow-up design](11-weighting-sizing-plan.md).

## Goal and method

Run the 12 registered local sensitivity combinations and eight constraint-aware allocation combinations, keeping regularised 90-day inverse variance as the benchmark. Then replay a small interpretable shortlist through the full trading engine. Keep the same six-vault focal_four and v11 selectors, their score recipes and missing=True behaviour. Determine whether nearby windows, prior strength, raw variance floors or caps inside allocation improve outcomes.

Sensitivities: two selectors × lookback 75/105 days; two selectors × kappa 10/90 actual updates at 90 days; two selectors × raw variance floors 1e-10/1e-6. Constraints: two selectors × equal/frozen-score risk budgets × projected/constrained convex allocations, with identical covariance, support and invested budget. The convex objective is variance minus calibrated weighted log diversification; it does not promise exact risk budgets under caps. All inherited numerical conditioning is recorded.

**2026-01-01 → 2026-10-05, exclusive end, UTC**, retaining all 2025 and earlier warmup. Snapshot origins every four days;7/30/90-day horizons. NAV alone is causally forward-filled, with no pre-launch backfill; actual observations remain distinct. Young vaults use partial empirical risk or the peer prior. Missing scores retain the frozen selection tier; zero score risk budgets get zero capital unless the entire target is zero. Unsupported terminal targets remain unresolved.

Engine replays use actual source decision/fill timing, two-day rebalancing, changing holdings, source hysteresis, operational deposit checks and applicable fees. Research sizing uses a common simultaneous cap projection; source v11 is a separate unchanged sequential-allocation control. Shortlist selection is retrospective and disclosed before reading engine results. There is no fresh holdout, and only eight complete primary 30-day blocks. Snapshot returns are not strategy CAGR.

## Key new insights

The original four-input score with source inverse-variance sizing and allocation remains strongest: **51.77% CAGR, 2.30 Sharpe, −6.35% maximum drawdown**. No replayed candidate beats it. A small sparse-estimator gain survives against the regularised benchmark, but adjacent-window and prior-strength snapshot improvements do not consistently survive full execution. Optimising with caps improves all four projected snapshot risk-budget pairs, yet trails the regularised inverse-variance benchmark; the engine cap effect is mixed.

Raw-floor changes are economically inactive on the selected baskets. The best snapshot sensitivities are 75 days for the four-input selector and kappa 90 for v11; engine CAGR changes are respectively +0.32 and −1.27 percentage points against their regularised 90-day benchmarks. This is a historical parameter comparison, not evidence of generalisation.

## Summary of results — 2026-01-01 → 2026-10-05

Completed **12 sensitivities + eight cap-aware policies + ten reproduced NB11 controls** over 70 origins and 7/30/90-day targets. Replayed 14 research/control policies plus two original strategy controls through the full engine. Full available 2026 evaluation history and earlier warmup are retained.

### Executable comparison

The table reports engine performance, including source holdings, trade thresholds and applicable fees. F means the four-input selector; L means v11 selection. Research allocations use a common simultaneous cap projection; original strategy controls retain their sequential allocator.

| Selection | Allocation | CAGR | Sharpe | Maximum drawdown |
| --- | --- | ---: | ---: | ---: |
| F | Original source sizing and allocator | **51.77%** | **2.30** | **−6.35%** |
| F | Regularised90-day inverse variance | 46.60% | 2.09 | −7.05% |
| F | Sparse gap-adjusted90-day inverse variance | 47.46% | 2.12 | −7.10% |
| F | Regularised75-day inverse variance | 46.92% | 2.12 | −7.08% |
| F | Constrained equal-risk allocation | 42.88% | 1.93 | −8.06% |
| L | Original v11 sizing and allocator | **41.20%** | **1.96** | **−6.40%** |
| L | Regularised90-day inverse variance | 33.34% | 1.63 | −6.60% |
| L | Four-component minimum score, T18 | 39.66% | 1.64 | −9.75% |
| L | Regularised90-day variance, kappa 90 | 32.07% | 1.53 | −7.06% |
| L | Constrained equal-risk allocation | 33.88% | 1.60 | −7.50% |

All 30 snapshot policies and all 16 engine runs are shown below. Snapshot means are independent-window returns and must not be compounded into CAGR.

## Robustness of results

Thirteen mathematical fixtures and 13 snapshot checks pass. All ten NB11 controls reproduce their weights and every horizon outcome. Constrained/projected pairs share covariance, support and invested budgets. Engine source controls reproduce the earlier repaired-data results; position PnL reconciles to equity within $1. Saved outputs have no errors or warnings; figures were individually inspected and corrected.

The sparse four-input gain over its regularised benchmark is only **$888**, with pmalt contributing +$1,606, Goon Edging +$968 and  Hyperrr −$1,683. The source strategies themselves have substantial profit concentration: Realist Capital represents about 38%/45% of positive vault PnL for F/L. The largest recorded update is a two-day valuation change, mainly Realist Capital and Goon Edging; actual daily NAV marks exist across the interval. These gains occurred while BTC/ETH fell, so a broad rising market does not explain them; the marks do not establish a data error.

The primary snapshot horizon contains only eight complete blocks; 90-day terminal support leaves one mature basket unresolved. Ranking tiers differ between the stateless snapshot and executable source’s zero-score ties, and the engine adds hysteresis, changing budgets, fees and operational checks. All results are retrospective and repeatedly searched. No production change is warranted by this experiment.

Every figure and table has its own heading and description; the default evaluation interval is 2026.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown
ROOT=Path.cwd()
if not (ROOT/'allocation_followup.py').exists():ROOT=ROOT/'scratchpad/hyperliquid-vov-2nd'
sys.path.insert(0,str(ROOT))
from allocation_followup import initialise, fixtures, registry, run_snapshots
from allocation_followup_report import snapshots, replay
from score_matrix_visualisation import compact_output
pd.options.display.max_colwidth=None


## Mathematical checks — 2026-01-01 → 2026-10-05

Independent examples check inverse-volatility risk budgets, cap feasibility, stationarity, covariance scaling, the diagonal minimum-variance identity and young/missing-history fallbacks.


In [2]:
display(fixtures())

,check,passed
0,Unconstrained diagonal equal risk is inverse volatility,True
1,Non-binding caps reproduce projected allocation,True
2,Binding cap objective improves,True
3,Binding cap feasibility and KKT,True
4,Covariance scale invariance,True
5,Diagonal minimum variance matches inverse variance water-fill,True
6,Missing empirical variance uses prior,True
7,Zero empirical variance stays distinct from missing,True
8,Raw floor sensitivities use declared floor,True
9,Young partial histories remain measurable,True


## Registered combinations and pinned inputs — 2026-01-01 → 2026-10-05

Twenty new combinations are accompanied by ten NB11 controls. This cell pins archive, source and policy hashes; earlier notebook outputs are preserved.


In [3]:
with compact_output('Verify pinned inputs; estimated under one minute'):
    data,policies=initialise()
display(pd.DataFrame(policies)[['arm','label','selector','method','L','kappa','floor']].style.hide(axis='index'))


Verify pinned inputs; estimated under one minute — complete

arm,label,selector,method,L,kappa,floor
S01,F:equal,F,equal,90,30.000000,nan
S02,L:equal,L,equal,90,30.000000,nan
S03,F:regular L90/p2/κ30,F,regular,90,30.000000,nan
S04,L:regular L90/p2/κ30,L,regular,90,30.000000,nan
S15,L:bounded_V,L,bounded_V,180,60.000000,nan
S16,L:T18,L,T18,90,30.000000,nan
S17,F:calendar_log L90/p2/κ30,F,calendar_log,90,30.000000,nan
S18,F:gap_log L90/p2/κ30,F,gap_log,90,30.000000,nan
M39,L:calendar_log L90/p2/κ30,L,calendar_log,90,30.000000,nan
M40,L:gap_log L90/p2/κ30,L,gap_log,90,30.000000,nan


## Full-calendar snapshots — 2026-01-01 → 2026-10-05; 7/30/90-day horizons

Allocate once per policy/origin, then value the three horizons independently. Exact NB11 controls, optimiser checks and shared-budget comparisons must pass before engine replay.


In [4]:
with compact_output('30 policies across 70 origins; estimated two–four minutes'):
    result=run_snapshots(data,policies)
snapshot_summary=snapshots(data,result)


### Snapshot results; 30-day horizon — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Every row holds a fixed six-vault basket for 30 days. Means use resolved origins only; these returns are not annualised strategy results. X arms are the 12 sensitivities, C arms the eight cap-aware methods; remaining rows reproduce NB11 controls.

Arm,Method,Origins,Mean 30-day return,Median return,Mean window drawdown,Cash,Effective positions
S01,F:equal,62,1.716%,1.115%,-6.193%,3.32%,5.19
S02,L:equal,62,1.929%,1.115%,-5.698%,3.01%,5.24
S03,F:regular L90/p2/κ30,62,2.512%,2.620%,-4.191%,3.32%,4.31
S04,L:regular L90/p2/κ30,62,1.992%,1.928%,-3.777%,3.01%,4.38
S15,L:bounded_V,62,2.031%,1.271%,-5.028%,3.01%,5.15
S16,L:T18,62,2.390%,1.271%,-5.003%,3.01%,5.07
S17,F:calendar_log L90/p2/κ30,62,2.413%,2.232%,-4.250%,3.32%,4.31
S18,F:gap_log L90/p2/κ30,62,2.579%,2.106%,-4.183%,3.32%,4.22
M39,L:calendar_log L90/p2/κ30,62,1.968%,1.816%,-3.782%,3.01%,4.37
M40,L:gap_log L90/p2/κ30,62,2.007%,1.726%,-3.831%,3.01%,4.29


### Reference reproduction and feasibility; 7/30/90-day horizons — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

All ten reused NB11 controls are recalculated. Accepted weights and future returns must reproduce their saved values; optimiser pairs must share the invested budget.

check,passed
S01: NB11 weights and all horizon outcomes,True
S02: NB11 weights and all horizon outcomes,True
S03: NB11 weights and all horizon outcomes,True
S04: NB11 weights and all horizon outcomes,True
S15: NB11 weights and all horizon outcomes,True
S16: NB11 weights and all horizon outcomes,True
S17: NB11 weights and all horizon outcomes,True
S18: NB11 weights and all horizon outcomes,True
M39: NB11 weights and all horizon outcomes,True
M40: NB11 weights and all horizon outcomes,True


### Optimiser mechanisms; all 70 scheduled origins — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

A solved allocation minimises the registered convex variance/diversification objective. Risk-contribution error need not fall when caps bind. Fallback status is explicit.

...

### L: prior strength sensitivity; 30-day horizon — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare 10,30 and 90 actual-update prior strengths at the same 90-day lookback.

### L: allocation trade-offs; 30-day horizon — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Blue: sensitivities; red: cap-aware methods; grey: controls; star: regularised 90-day benchmark. Compare methods within a selector. Arm names are listed in the results table.

### Constraint-aware paired effects by quarter; 30-day horizon — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each comparison shares covariance, risk-budget target, support and invested budget. Positive cells favour solving with caps inside the objective. Q4 has no mature 30-day origin.

### Secondary horizons; 7 and 90 days — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Allocations are identical across horizons. The 90-day comparison has sparse independent support and one unresolved administratively mature basket; retain those limitations.

arm,h,mature,resolved,mean_return,mean_drawdown
S01,7,68,68,0.679%,-2.128%
S01,90,47,46,5.449%,-13.714%
S02,7,68,68,0.562%,-2.063%
S02,90,47,46,5.225%,-13.135%
S03,7,68,68,0.877%,-1.642%
S03,90,47,46,3.791%,-11.547%
S04,7,68,68,0.588%,-1.576%
S04,90,47,46,3.390%,-10.721%
S15,7,68,68,0.588%,-1.880%
S15,90,47,46,4.736%,-12.182%


## Engine shortlist and replay — 2026-01-01 → 2026-10-05

Keep equal and regularised 90-day controls for both selectors; matched sparse controls; T18 and its V180 control; both equal-risk cap methods for both selectors; one sensitivity per selector chosen for its mean 30-day return; and unchanged source v11. The sensitivity choice is exploratory. Freeze these identities before observing engine results.


In [5]:
from allocation_followup_engine import run_shortlist
with compact_output('Full engine shortlist; estimated three–eight minutes'):
    engine_summary,runs,robustness=run_shortlist(data,result,policies)
replay(data,engine_summary,runs,robustness)


Full engine shortlist; estimated three–eight minutes — complete

### Full engine results; two-day rebalancing — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Actual engine runs start with $150,000, retain source hysteresis, entry/exit checks, trade thresholds and applicable redemption fees. Research allocations use a common simultaneous cap projection; the source v11 control retains its original sequential allocator. CAGR below belongs to the engine runs.

Arm,Method,CAGR,Sharpe,Sortino,Maximum drawdown,Final equity (USD),Mean cash weight,Trades
S01,F:equal,37.97%,1.53,2.38,-11.58%,"$191,142",9.39%,582
S02,L:equal,33.63%,1.38,2.14,-10.56%,"$186,582",8.37%,584
S03,F:regular L90/p2/κ30,46.60%,2.09,3.62,-7.05%,"$200,120",9.68%,598
S04,L:regular L90/p2/κ30,33.34%,1.63,2.72,-6.60%,"$186,274",8.33%,606
S15,L:bounded_V,34.25%,1.46,2.29,-9.95%,"$187,231",8.39%,609
S16,L:T18,39.66%,1.64,2.63,-9.75%,"$192,913",8.64%,612
S17,F:calendar_log L90/p2/κ30,47.04%,2.12,3.68,-7.08%,"$200,576",9.72%,606
S18,F:gap_log L90/p2/κ30,47.46%,2.12,3.72,-7.10%,"$201,008",9.73%,604
C01,F:equal_risk / projected,42.98%,1.92,3.19,-8.26%,"$196,376",9.60%,600
C02,F:equal_risk / constrained_convex,42.88%,1.93,3.22,-8.06%,"$196,265",9.56%,596


### F: engine equity curves — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each curve carries holdings through time. The results table maps arm IDs to methods. Compare against the same selector’s regularised 90-day benchmark.

### F: benchmark absolute weights — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Weights include cash and sum to the whole account. The ten vaults with highest average exposure are named; all remaining vaults are grouped as Other. Absent vaults are zero at observed timestamps, so closed holdings are not forward-filled. At most six vaults are active; the complete weights table is saved.

### L: engine equity curves — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each curve carries holdings through time. The results table maps arm IDs to methods. Compare against the same selector’s regularised 90-day benchmark.

### L: benchmark absolute weights — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Weights include cash and sum to the whole account. The ten vaults with highest average exposure are named; all remaining vaults are grouped as Other. Absent vaults are zero at observed timestamps, so closed holdings are not forward-filled. At most six vaults are active; the complete weights table is saved.

### Tail and vault influence; full engine runs — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Positive-vault concentration is the largest vault’s final PnL divided by all positive vault PnL. Best-day contribution and kurtosis describe return concentration; attribution is reconciled to portfolio PnL.

Arm,Best day,Worst day,Best day date,Excess kurtosis,Top positive vault,Top positive vault share,PnL residual (USD)
S01,6.84%,-4.94%,2026-06-26,7.415235,Realist Capital,35.3%,0.000
S02,6.87%,-5.06%,2026-06-26,7.349009,Realist Capital,39.8%,0.000
S03,6.81%,-4.72%,2026-06-26,11.133952,Realist Capital,37.8%,0.000
S04,7.05%,-5.06%,2026-06-26,13.315592,Realist Capital,40.0%,0.000
S15,6.98%,-5.04%,2026-06-26,8.231274,Realist Capital,40.9%,-0.000
S16,6.97%,-4.90%,2026-06-26,8.026562,Realist Capital,42.1%,0.000
S17,6.81%,-4.71%,2026-06-26,11.252963,Realist Capital,37.8%,0.000
S18,6.80%,-4.70%,2026-06-26,11.333579,Realist Capital,37.9%,0.000
C01,6.82%,-4.81%,2026-06-26,10.202296,Realist Capital,38.2%,0.000
C02,6.82%,-4.81%,2026-06-26,10.461157,Realist Capital,38.0%,0.000


### Engine quarterly profit heatmap — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Each cell compounds that run’s daily equity returns within the quarter. Q4 covers only the available October days. This differs from averaging independent snapshot returns.

## Findings and robustness — 2026-01-01 → 2026-10-05

Write the verdict after successful snapshot checks, complete engine execution and position-level attribution. Report searched-history limitations and the differences between snapshot and executable allocation mechanics.


In [6]:
from allocation_followup_engine import write_findings
findings=write_findings(data,result,engine_summary,runs,robustness)
from allocation_followup_review import render_review
engine_contrasts=render_review(data)
display(Markdown(f'[Detailed findings]({findings.name})'))


### Engine improvements against both controls — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Compare the highest observed candidate CAGR per selector with its regularised benchmark and original strategy control. This selection is retrospective; it is a diagnostic, not validation.

Selector,Arm,Control,CAGR difference (pp),Sharpe difference,Drawdown difference (pp),Equity difference (USD)
F,S18,S03,+0.86,+0.03,-0.06,$+888
F,S18,source_focal,-4.31,-0.18,-0.76,"$-4,422"
L,S16,S04,+6.32,+0.01,-3.15,"$+6,639"
L,S16,source_v11,-1.54,-0.32,-3.34,"$-1,608"


### Vault contributions to candidate improvements — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Largest positive and negative PnL differences are shown for each comparison. Removing a contribution is an influence diagnostic; it does not recreate a portfolio without that vault. Full attribution reconciles to the equity difference.

Selector,Arm,Control,Vault,PnL difference (USD),Total equity difference (USD)
L,S16,S04,Hyperrr,"$+4,909","$+6,639"
L,S16,S04,Realist Capital,"$+3,696","$+6,639"
L,S16,S04,pmalt,"$-3,853","$+6,639"
L,S16,S04,AceVault Hyper01,$-871,"$+6,639"
L,S16,source_v11,Hyperrr,"$+4,726","$-1,608"
L,S16,source_v11,Citadel,"$+1,599","$-1,608"
L,S16,source_v11,pmalt,"$-4,837","$-1,608"
L,S16,source_v11,Realist Capital,"$-1,814","$-1,608"
F,S18,S03,pmalt,"$+1,606",$+888
F,S18,S03,Goon Edging,$+968,$+888


### Largest individual position profits — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Show the two largest position PnLs for each control and highest observed candidate. These are marked cash-flow profits, including still-open positions, rather than only realised trades.

Arm,Position id,Vault,Opened,Closed,Final profit (USD)
S03,67,Realist Capital,2026-06-20 00:00:00,2026-08-21 00:00:00,"$24,314"
S03,101,DOEZOE,2026-09-18 00:00:00,NaT,"$11,815"
S04,64,Realist Capital,2026-06-20 00:00:00,2026-08-21 00:00:00,"$18,316"
S04,98,DOEZOE,2026-09-18 00:00:00,NaT,"$11,815"
S16,64,Realist Capital,2026-06-20 00:00:00,2026-08-21 00:00:00,"$21,828"
S16,98,DOEZOE,2026-09-18 00:00:00,NaT,"$11,815"
S18,67,Realist Capital,2026-06-20 00:00:00,2026-08-21 00:00:00,"$24,445"
S18,101,DOEZOE,2026-09-18 00:00:00,NaT,"$11,815"
source_focal,67,Realist Capital,2026-06-20 00:00:00,2026-08-21 00:00:00,"$26,168"
source_focal,101,DOEZOE,2026-09-18 00:00:00,NaT,"$11,815"


[Detailed findings](12-allocation-followup-findings.md)

## Tail attribution and valuation clock — 2026-01-01 → 2026-10-05

Check the largest recorded equity update against position cash-flow PnL, actual archive NAV marks, current curator exclusions and BTC/ETH reference prices. Daily displayed returns contain the engine’s two-day valuation updates; large recorded changes are not independently observed one-day portfolio returns.

In [7]:
from allocation_tail_audit import audit_tail
tail_audit=audit_tail(data)
(folder if 'folder' in globals() else data['artifact']).joinpath('figure-manifest.json').write_text(__import__('json').dumps(list(dict.fromkeys(data['figures'])),indent=2));

### Largest recorded equity update — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

The equity series is marked at the engine’s two-day cadence. Its largest non-zero return is recorded on 26 June and spans multiple days; it should not be interpreted as a independently observed one-day return. Realist Capital and Goon Edging account for most of the PnL increase. Actual positive archive marks exist on each of 23–25 June; the BTC/ETH comparison uses cached Binance daily prices. These gains occurred while both references fell, so a broad rising market does not explain them. This does not establish an archive error.

Asset,23→25 June NAV return,Actual daily marks,Recorded PnL change (USD),Curator excluded,Quarantined at update
Realist Capital,+14.06%,3,"$6,930",False,False
Goon Edging,+13.86%,3,"$4,620",False,False
BTCUSDT,-4.69%,3,Not held,False,False
ETHUSDT,-5.96%,3,Not held,False,False


### Source v11 profit concentration — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Closed positions retain realised PnL. Profits are aggregated across entries in each vault and reconcile to total portfolio equity less initial capital. This is an influence diagnostic rather than a rerun excluding these vaults.

### Vault PnL on the largest recorded update — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

The two largest contributions are about $6,930 from Realist Capital and $4,620 from Goon Edging. Position cash flows remove the effect of purchases and sales from profit attribution; all figures use the same recorded valuation clock.